In [5]:
# السطر هذا يقوم بتثبيت المكتبة الناقصة مباشرة في جوبيتر
%pip install sentence-transformers

   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.9 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.9 kB ? eta -:--:--
   --------------------------------- ---- 524.3/588.9 kB 634.0 kB/s eta 0:00:01
   ---------------------------------------- 588.9/588.9 kB 541.1 kB/s  0:00:01
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
    --------------------------------------- 0.3/11.2 MB ? eta -:--:--
    --------------------------------------- 0.3/11.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.2 MB 547.3 kB/s eta 0:00:20
   -- ------------------------------------- 0.8/11.2 MB 850.0 kB/s eta 0:00:13
   -- ------------------------------------- 0.8/11.2 MB 850.0 kB/s eta 0:00:13
   -- -----------------------------

In [9]:
import numpy as np
import pandas as pd
import os

# =====================================================================
# 1. استدعاء الخدمات من كافة الملفات
# =====================================================================
from retrieval_service import build_inverted_index, prepare_bm25, bm25_search
from embedding_service import SemanticSearchService
from lama_evaluation_rag_service import EvaluationService, RAGService

print("⚙️ جاري تجهيز النظام بالكامل...")

# بناء الفهرس لـ BM25
inverted_index = build_inverted_index(df)
bm25_idf, avg_doc_length = prepare_bm25(df, inverted_index)

# تهيئة خدمة الـ BERT (بدون تحميل أشعة قديمة لأننا سنحسب مباشرة للوثائق المسترجعة)
bert_service = SemanticSearchService()
bert_service.initialize_service(load_embeddings=False)

# تهيئة خدمات التقييم والـ RAG
eval_service = EvaluationService()
rag_service = RAGService()

print("✅ تم تحميل وتجهيز كافة الخدمات بنجاح!")


# =====================================================================
# 2. بناء تابع البحث التسلسلي المشترك (Cascade Search Function)
# =====================================================================
def cascade_search_pipeline(query, top_n_bm25=50, top_k_final=5):
    """
    المرحلة الأولى: جلب أعلى N وثيقة باستخدام BM25 السريع.
    المرحلة الثانية: إعادة ترتيب هذه الـ N وثيقة فقط باستخدام نموذج BERT.
    """
    # 1. البحث باستخدام BM25
    bm25_results = bm25_search(query, df, inverted_index, bm25_idf, avg_doc_length)
    top_bm25_candidates = bm25_results[:top_n_bm25] # أخذ أعلى 50 وثيقة مثلاً
    
    if not top_bm25_candidates:
        return []
        
    # استخراج المعرفات والنصوص للوثائق المرشحة فقط
    candidate_ids = [doc_id for doc_id, _ in top_bm25_candidates]
    candidate_df = df[df["doc_id"].isin(candidate_ids)].copy()
    candidate_texts = candidate_df["cleaned_text"].astype(str).tolist()
    
    # 2. إعادة الترتيب بواسطة BERT
    query_embedding = bert_service.model.encode([query])
    doc_embeddings = bert_service.model.encode(candidate_texts, show_progress_bar=False)
    
    # حساب التشابه لجيب التمام (Cosine Similarity)
    from sentence_transformers import util
    scores = util.cos_sim(query_embedding, doc_embeddings)[0].tolist()
    
    # ربط المعرفات بالسكور الجديد والترتيب تنازلياً
    ranked_results = sorted(
        zip(candidate_df["doc_id"].tolist(), candidate_texts, scores),
        key=lambda x: x[2],
        reverse=True
    )
    
    return ranked_results[:top_k_final]


# =====================================================================
# 3. تشغيل السيناريو الكامل (End-to-End Demo)
# =====================================================================
print("\n🚀 --- تشغيل محاكاة البحث والـ RAG والتقييم ---")

# استعلام تجريبي
user_query = "how to learn machine learning"
print(f"🔍 استعلام المستخدم: '{user_query}'")

# تشغيل الـ Pipeline المشترك
final_results = cascade_search_pipeline(user_query, top_n_bm25=20, top_k_final=5)

# عرض النتائج المسترجعة النهائية
print("\n🔝 أفضل 3 وثائق مسترجعة بعد إعادة الترتيب بـ BERT:")
retrieved_ids_for_eval = []
retrieved_texts_for_rag = []

for rank, (doc_id, text, score) in enumerate(final_results[:3], 1):
    print(f"المرتبة {rank} | معرف الوثيقة: {doc_id} | السكور الدلالي: {round(score, 4)}")
    print(f"📄 النص: {text}\n" + "-"*40)
    retrieved_ids_for_eval.append(doc_id)
    retrieved_texts_for_rag.append(text)

# تنفيذ ميزة الـ RAG الذكية بناءً على هذه النتائج
rag_answer = rag_service.generate_smart_answer(user_query, retrieved_texts_for_rag)
print(f"\n🤖 إجابة نظام الـ RAG التوليدي:\n{rag_answer}")


# =====================================================================
# 4. تقييم جودة النظام (Evaluation)
# =====================================================================
print("\n📊 --- تقييم جودة الاسترجاع ---")

# سنفترض جدلاً أن هذه هي المعرفات الحقيقية المطلوبة للاختبار (Ground Truth)
mock_relevant_ids = [final_results[0][0], final_results[1][0], 999] 

# حساب المقاييس
metrics = eval_service.evaluate_model(
    all_retrieved=[[doc_id for doc_id, _, _ in final_results]], 
    all_relevant=[mock_relevant_ids], 
    k=5
)

print("📝 المقاييس المحسوبة للنظام التسلسلي:")
for metric_name, value in metrics.items():
    print(f"🔹 {metric_name}: {value}")

⚙️ جاري تجهيز النظام بالكامل...
====== [Initialization] ======
Loading Model: sentence-transformers/all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\dell\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully.
Reading Data from: quora_cleaned_sample.csv...
Data Loaded Successfully. Shape: (9990, 3)
RAG Service Initialized Successfully.
✅ تم تحميل وتجهيز كافة الخدمات بنجاح!

🚀 --- تشغيل محاكاة البحث والـ RAG والتقييم ---
🔍 استعلام المستخدم: 'how to learn machine learning'

🔝 أفضل 3 وثائق مسترجعة بعد إعادة الترتيب بـ BERT:
المرتبة 1 | معرف الوثيقة: 2716 | السكور الدلالي: 0.601
📄 النص: best way learning
----------------------------------------
المرتبة 2 | معرف الوثيقة: 6627 | السكور الدلالي: 0.5963
📄 النص: machine learning beginner go straight deep learning
----------------------------------------
المرتبة 3 | معرف الوثيقة: 249 | السكور الدلالي: 0.5868
📄 النص: alternative machine learning
----------------------------------------

🤖 إجابة نظام الـ RAG التوليدي:
[RAG Answer] summary based on the retrieved data for your query: 'how to learn machine learning'.

📊 --- تقييم جودة الاسترجاع ---
📝 المقاييس المحسوبة للنظام التسلسلي:
🔹 Precision@5: 0.4
🔹 Recall: 0.6667
🔹 nDCG@5: